# Per-Muscle and Overall Average Metrics — Asian Dataset Algorithms

Reads per-muscle result CSVs from each algorithm's `results_asian_water/` folder,
computes the mean of every metric across all subjects, appends an `Overall_Mean` row,
and saves a summary CSV.  The final cell combines all `Overall_Mean` rows into one
comparison table.

In [ ]:
import pathlib
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

_FLOAT64_MAX = np.finfo(np.float64).max

# Metric suffixes in priority order (longer / more specific first).
KNOWN_METRICS = [
    'inter_slice_dice_pred',
    'inter_slice_dice_gt',
    'boundary_iou_3d',
    'volume_similarity',
    'false_negative',
    'false_positive',
    'hausdorff',
    'jaccard',
    'dice',
    'bce',
]

def canonical_metric(col):
    """Return the metric name if col ends with a known metric suffix, else None."""
    c = col.lower()
    for metric in KNOWN_METRICS:
        if c == metric or c.endswith('_' + metric):
            return metric
    return None


def extract_muscle(df):
    """
    Extract the muscle-name prefix straight from the dataframe's own *_dice
    column (e.g. 'gracilis_L_dice' -> 'gracilis_L', 'L_gracilis_dice' ->
    'L_gracilis'). Works regardless of filename/algorithm-tag naming
    convention, unlike parsing it out of the CSV filename.
    """
    dice_cols = [c for c in df.columns if c.endswith('_dice')]
    if not dice_cols:
        return None
    return dice_cols[0][:-len('_dice')]


def is_left_side(muscle_name):
    """
    True if this muscle name is explicitly tagged as the left side
    ('gracilis_L' or 'L_gracilis' style). The Asian dataset's ground truth
    only has right-side labels, so a left-tagged prediction is being
    compared against an empty/absent GT — scoring it would just measure
    "how much did the model predict on a side with nothing there", which
    unfairly drags down algorithms that happen to predict both sides versus
    ones that only ever output a single combined (right-only-meaningful)
    muscle. Left-tagged muscles are excluded from averaging entirely below,
    not just down-weighted.
    """
    return muscle_name.endswith('_L') or muscle_name.startswith('L_')


def process_algorithm(label, results_dir):
    """Return a summary DataFrame for one algorithm, or None if no CSVs found."""
    results_dir = pathlib.Path(results_dir)
    csv_files   = sorted(results_dir.glob('df_*.csv'))   # only per-muscle files

    if not csv_files:
        print(f'  [skip] no CSVs in {results_dir}')
        return None

    rows = []
    for csv_path in csv_files:
        df = pd.read_csv(csv_path, index_col=0)
        muscle = extract_muscle(df)
        if muscle is None:
            print(f'  [skip] could not identify muscle in {csv_path.name}')
            continue
        if is_left_side(muscle):
            print(f'  [skip left-side, no GT] {csv_path.name}')
            continue

        metric_vals = {}

        for col in df.columns:
            metric_name = canonical_metric(col)
            if metric_name is None:
                continue

            s = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=np.float64)
            s = np.where(np.isfinite(s) & (np.abs(s) < _FLOAT64_MAX), s, np.nan)

            finite = np.isfinite(s)
            if not finite.any():
                print(f'  [ALL NON-FINITE] {csv_path.name} | {col}')
                continue

            metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)

        # Derived: inter-slice dice ratio
        pred = metric_vals.get('inter_slice_dice_pred')
        gt   = metric_vals.get('inter_slice_dice_gt')
        if pred is not None and gt is not None and gt > 0:
            metric_vals['inter_slice_dice_ratio'] = pred / gt

        row = {'muscle': muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index('muscle')
    summary.insert(0, 'algorithm', label)

    numeric = summary.select_dtypes(include='number').astype(np.float64)
    overall = numeric.mean().rename('Overall_Mean')
    overall['algorithm'] = label
    summary = pd.concat([summary, overall.to_frame().T])
    summary.index.name = 'muscle'
    return summary


print('Helpers ready.')

In [ ]:
# ── Algorithm registry ────────────────────────────────────────────────────────
# Add new algorithms here as results_asian* folders are populated.

EVAL_DIR    = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')
SUMMARY_DIR = EVAL_DIR / 'summary_results_asian'
SUMMARY_DIR.mkdir(exist_ok=True)

REGISTRY = [
    ('Dafne (water)',                      'dafne/codes/results_asian_water'),
    ('MuscleMap WB (water)',               'muscle_map_wb/codes/results_asian_water'),
    ('MuscleMap Thigh (water)',            'muscle_map_thigh/codes/results_asian_water'),
    ('MuSeg (Dixon)',                      'museg/codes/results_asian_dixon_based'),
    ('MuSeg (water only)',                 'museg/codes/results_asian_water_only'),
    ('Hirriririir (water)',                'multimodal-multiethnic/codes/results_asian_water'),
    ('MedSegDiff',                         'medsegdiff/codes/results_asian'),
    ('MedCLIP-SAMv2 Text+Boxes (water)',   'medclipsamv2textboxes/codes/results_asian_water'),
]

print(f'{len(REGISTRY)} entries in registry.')
for label, rdir in REGISTRY:
    path = EVAL_DIR / rdir
    n = len(list(path.glob('df_*.csv'))) if path.exists() else 0
    status = f'{n} CSVs' if path.exists() else 'DIR MISSING'
    print(f'  {label}: {status}')

In [ ]:
# ── Process all algorithms ────────────────────────────────────────────────────

summaries = {}

for label, rdir in REGISTRY:
    print(f'\n── {label} ──')
    df = process_algorithm(label, EVAL_DIR / rdir)
    if df is None:
        continue
    summaries[label] = df

    num_cols = df.select_dtypes(include='number').columns
    display(df.reset_index().style.format('{:.4f}', subset=num_cols).hide(axis='index'))

    safe_name = re.sub(r'[^\w]+', '_', label).strip('_').lower()
    out_path  = SUMMARY_DIR / f'{safe_name}_avg_metrics.csv'
    df.to_csv(out_path, float_format='%.4f')
    print(f'  Saved -> {out_path}')

print(f'\nProcessed {len(summaries)}/{len(REGISTRY)} algorithms.')

In [ ]:
# ── Combined Overall Means ────────────────────────────────────────────────────

overall_rows = [
    df.loc[['Overall_Mean']]
    for df in summaries.values()
    if 'Overall_Mean' in df.index
]

combined = pd.concat(overall_rows)
combined.index = [row['algorithm'] for _, row in combined.iterrows()]
combined.index.name = 'algorithm'
combined = combined.drop(columns='algorithm')
combined = combined.drop(
    columns=['inter_slice_dice_pred', 'inter_slice_dice_gt'], errors='ignore'
)

num_cols = combined.select_dtypes(include='number').columns
display(
    combined.reset_index()
    .style
    .format('{:.4f}', subset=num_cols)
    .hide(axis='index')
    .background_gradient(subset=['dice'],       cmap='RdYlGn',   axis=0)
    .background_gradient(subset=['hausdorff'],  cmap='RdYlGn_r', axis=0)
)

out_combined = SUMMARY_DIR / 'overall_means_asian.csv'
combined.to_csv(out_combined, float_format='%.4f')
print('Saved ->', out_combined)